# 📈 Model 2 — Demand Forecasting (v2)
### ARIMA · Holt-Winters ETS · Prophet
**Input:** `data_ts_weekly.csv` · **Output:** `data_forecast.csv` + `models/`

> Three models compared on 12-week hold-out · All include confidence intervals · Best model wins

**Why Holt-Winters instead of SARIMA?**  
SARIMA(m=52) requires ≥3 full annual cycles to reliably estimate seasonal parameters.  
Our dataset has only ~1.8 cycles (106 weeks ÷ 52). Holt-Winters exponential smoothing  
learns the seasonal pattern from a single complete cycle — ideal for 2 years of data.

In [ ]:
!pip install prophet pmdarima statsmodels --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima

C = ['#2C6FAC', '#27AE60', '#E74C3C', '#F39C12', '#8E44AD']
plt.rcParams.update({
    'figure.figsize'    : (15, 5),
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'font.size'         : 11,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
})
print('✓ Libraries ready')

## Cell 2 — Load Weekly Time Series

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE = '/content/drive/MyDrive/retail_project/'

ts = pd.read_csv(SAVE + 'data_ts_weekly.csv', parse_dates=['ds'])
ts = ts.sort_values('ds').reset_index(drop=True)
ts['y'] = ts['y'].clip(lower=0)

print(f'Shape       : {ts.shape}')
print(f'Date range  : {ts["ds"].min().date()} → {ts["ds"].max().date()}')
print(f'Weeks       : {len(ts)}')
print(f'Avg revenue : £{ts["y"].mean():,.0f}/week')
print(f'Peak revenue: £{ts["y"].max():,.0f}/week  ← Christmas spike')
print(f'Min revenue : £{ts["y"].min():,.0f}/week')
ts.head(5)

## Cell 3 — Visualize Raw Series

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Weekly Revenue — Raw Time Series', fontsize=14, fontweight='bold')

# Top: full series + rolling averages
axes[0].fill_between(ts['ds'], ts['y']/1000, alpha=0.12, color=C[0])
axes[0].plot(ts['ds'], ts['y']/1000, color=C[0], lw=1, alpha=0.6, label='Weekly revenue')
axes[0].plot(ts['ds'], ts['y'].rolling(4).mean()/1000, color=C[1], lw=2, label='4-week avg')
axes[0].plot(ts['ds'], ts['y'].rolling(12).mean()/1000, color=C[2], lw=2.5, label='12-week avg')

for yr in [2009, 2010, 2011]:
    axes[0].axvspan(
        pd.Timestamp(f'{yr}-11-01'), pd.Timestamp(f'{yr}-12-31'),
        alpha=0.12, color=C[2],
        label='Nov-Dec (Christmas)' if yr == 2009 else '_nolegend_'
    )

axes[0].set_title('Full Weekly Revenue + Moving Averages')
axes[0].set_ylabel('Revenue (£K)')
axes[0].legend(loc='upper left')
axes[0].tick_params(axis='x', rotation=30)

colors_bar = [C[2] if m in [11, 12] else C[0] for m in ts['ds'].dt.month]
axes[1].bar(range(len(ts)), ts['y']/1000, color=colors_bar, alpha=0.75, width=1)
axes[1].set_title('Revenue by Week — Blue=Normal  Red=Christmas (Nov-Dec)')
axes[1].set_ylabel('Revenue (£K)')
axes[1].set_xlabel('Week #')

plt.tight_layout()
plt.show()

## Cell 4 — STL Seasonal Decomposition (NEW)
> STL splits the series into Trend + Seasonal + Residual.  
> This confirms **exactly** when the Christmas peak occurs and how large it is —  
> essential for understanding what our models need to capture.

In [ ]:
ts_series = ts.set_index('ds')['y']

stl     = STL(ts_series, period=52, robust=True)
stl_res = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle('STL Decomposition — Annual Seasonality (period = 52 weeks)',
             fontsize=14, fontweight='bold')

components = [
    (ts_series/1000,           'Observed (£K)',          C[0]),
    (stl_res.trend/1000,       'Trend (£K)',              C[1]),
    (stl_res.seasonal/1000,    'Seasonal Component (£K)', C[2]),
    (stl_res.resid/1000,       'Residual (£K)',           C[3]),
]

for ax, (data, label, color) in zip(axes, components):
    ax.plot(data.index, data, color=color, lw=1.5)
    if 'Seasonal' in label or 'Residual' in label:
        ax.axhline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_ylabel(label, fontsize=9)
    ax.tick_params(axis='x', rotation=30)

# Mark seasonal peak
seasonal_peak_wk = stl_res.seasonal.idxmax()
axes[2].axvline(seasonal_peak_wk, color='black', lw=1.5, linestyle=':')
axes[2].annotate(
    f'Peak: {seasonal_peak_wk.strftime("%b %Y")}',
    xy=(seasonal_peak_wk, stl_res.seasonal.max()/1000),
    xytext=(10, -20), textcoords='offset points', fontsize=9
)

plt.tight_layout()
plt.show()

peak_val   = stl_res.seasonal.max()
trough_val = stl_res.seasonal.min()
print(f'Seasonal peak    : +£{peak_val:,.0f}/week  ({seasonal_peak_wk.strftime("%b %Y")})')
print(f'Seasonal trough  : £{trough_val:,.0f}/week')
print(f'Peak-to-trough   : £{peak_val - trough_val:,.0f}/week swing')
print()
print('→ ARIMA ignores this entirely → flat forecast')
print('→ Holt-Winters & Prophet must capture this spike to be useful')

## Cell 5 — Train / Test Split
> Last **12 weeks** = test (held out). Train on everything before.

In [ ]:
TEST_WEEKS = 12

train = ts.iloc[:-TEST_WEEKS].copy()
test  = ts.iloc[-TEST_WEEKS:].copy()

print(f'Train : {len(train)} weeks  {train["ds"].min().date()} → {train["ds"].max().date()}')
print(f'Test  : {len(test)}  weeks  {test["ds"].min().date()} → {test["ds"].max().date()}')
xmas_in_test = test['ds'].dt.month.isin([11,12]).sum()
print(f'Christmas weeks in test: {xmas_in_test} — models must predict these correctly')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train['ds'], train['y']/1000, color=C[0], lw=2, label=f'Train ({len(train)} weeks)')
ax.plot(test['ds'],  test['y']/1000,  color=C[2], lw=2, label=f'Test  ({len(test)} weeks)')
ax.axvline(test['ds'].iloc[0], color='black', lw=1.5, linestyle='--', label='Split point')
ax.set_title('Train / Test Split')
ax.set_ylabel('Revenue (£K)')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Cell 6 — Evaluation Helper

In [ ]:
def evaluate(actual, predicted, model_name, verbose=True):
    actual    = np.array(actual)
    predicted = np.array(predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    mask = actual > 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100
    if verbose:
        print(f'  {model_name:<20}  RMSE: £{rmse:>10,.0f}  |  MAE: £{mae:>10,.0f}  |  MAPE: {mape:>6.1f}%')
    return {'model': model_name, 'rmse': rmse, 'mae': mae, 'mape': mape}

results = {}
print('✓ Evaluation function ready')

## Cell 7 — Model A: ARIMA
> Baseline model — captures trend, **no seasonality**.  
> Expected: flat or slowly declining forecast — will miss the Christmas spike entirely.

In [ ]:
print('⏳ Searching best ARIMA(p,d,q) — ~1 min ...')

arima_model = auto_arima(
    train['y'],
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    information_criterion='aic',
    max_p=5, max_q=5, max_d=2,
)

print(f'✓ Best ARIMA order : {arima_model.order}')
print(f'  AIC              : {arima_model.aic():.2f}')

# Point forecast + 95% confidence interval
arima_pred, arima_ci = arima_model.predict(
    n_periods=TEST_WEEKS, return_conf_int=True, alpha=0.05
)
arima_pred = np.clip(arima_pred, 0, None)
arima_ci   = np.clip(arima_ci, 0, None)

print(f'\nTest set evaluation:')
results['ARIMA'] = evaluate(test['y'], arima_pred, 'ARIMA')
results['ARIMA'].update({
    'predictions': arima_pred,
    'lower'      : arima_ci[:, 0],
    'upper'      : arima_ci[:, 1],
    'model_obj'  : arima_model,
})

## Cell 8 — Model B: Holt-Winters ETS
> **Replaces SARIMA.**  
> Holt-Winters exponential smoothing with multiplicative annual seasonality.  
> Works from a **single complete seasonal cycle** — perfect for 2 years of data.  
> SARIMA(m=52) needs ≥3 full cycles; we only have ~1.8 → that's why it gave flat results.

In [ ]:
print('⏳ Fitting Holt-Winters ETS (trend=add, seasonal=mul, m=52) ...')

hw_model = ExponentialSmoothing(
    train['y'],
    trend='add',
    seasonal='mul',          # multiplicative: Christmas spike scales with trend level
    seasonal_periods=52,
    damped_trend=True,       # prevents runaway extrapolation beyond training range
    initialization_method='estimated',
)
hw_fit = hw_model.fit(optimized=True, remove_bias=True)

# Point forecast
hw_pred = hw_fit.forecast(TEST_WEEKS).values
hw_pred = np.clip(hw_pred, 0, None)

# 95% Confidence Interval via bootstrap simulation (500 paths)
np.random.seed(42)
sims     = hw_fit.simulate(TEST_WEEKS, repetitions=500, error='mul')
hw_lower = np.clip(np.percentile(sims, 2.5,  axis=1), 0, None)
hw_upper = np.clip(np.percentile(sims, 97.5, axis=1), 0, None)

alpha = hw_fit.params['smoothing_level']
beta  = hw_fit.params.get('smoothing_trend', 0)
gamma = hw_fit.params.get('smoothing_seasonal', 0)
print(f'✓ HW fitted  α(level)={alpha:.3f}  β(trend)={beta:.3f}  γ(seasonal)={gamma:.3f}')

print(f'\nTest set evaluation:')
results['Holt-Winters'] = evaluate(test['y'], hw_pred, 'Holt-Winters')
results['Holt-Winters'].update({
    'predictions': hw_pred,
    'lower'      : hw_lower,
    'upper'      : hw_upper,
    'model_obj'  : hw_fit,
})

## Cell 9 — Model C: Prophet
> Facebook/Meta's forecasting library.  
> Uses Fourier series for seasonality → no need for multiple full annual cycles.  
> Incorporates UK bank holidays + tuned changepoint & seasonality priors.

In [ ]:
from prophet import Prophet

print('⏳ Training Prophet model ...')

prophet_model = Prophet(
    yearly_seasonality      = True,
    weekly_seasonality      = False,    # B2B wholesaler — no day-of-week pattern
    daily_seasonality       = False,
    seasonality_mode        = 'multiplicative',  # seasonal effect scales with trend
    interval_width          = 0.95,
    changepoint_prior_scale = 0.15,    # allows flexible trend changes (default 0.05 too rigid)
    seasonality_prior_scale = 20,      # stronger annual seasonality signal
    n_changepoints          = 20,
)
prophet_model.add_country_holidays(country_name='GB')
prophet_model.fit(train[['ds', 'y']])

# Build future dataframe covering full train + test period
future        = prophet_model.make_future_dataframe(periods=TEST_WEEKS, freq='W')
forecast_full = prophet_model.predict(future)

# Align predictions to test dates exactly
forecast_test = forecast_full[forecast_full['ds'].isin(test['ds'])].copy()
if len(forecast_test) < TEST_WEEKS:
    forecast_test = forecast_full.tail(TEST_WEEKS)  # fallback

prophet_pred  = np.clip(forecast_test['yhat'].values,       0, None)
prophet_lower = np.clip(forecast_test['yhat_lower'].values, 0, None)
prophet_upper = np.clip(forecast_test['yhat_upper'].values, 0, None)

print(f'✓ Prophet fitted | changepoint scale=0.15 | seasonality scale=20')
print(f'\nTest set evaluation:')
results['Prophet'] = evaluate(test['y'], prophet_pred, 'Prophet')
results['Prophet'].update({
    'predictions'  : prophet_pred,
    'lower'        : prophet_lower,
    'upper'        : prophet_upper,
    'model_obj'    : prophet_model,
    'forecast_full': forecast_full,
    'forecast_test': forecast_test,
})

## Cell 10 — Model Comparison

In [ ]:
print('=' * 65)
print('  MODEL COMPARISON — Test Set (12 weeks)')
print('=' * 65)
print(f"  {'Model':<18} {'RMSE':>12} {'MAE':>12} {'MAPE':>8}")
print('-' * 65)
for name, res in results.items():
    print(f"  {name:<18} £{res['rmse']:>10,.0f} £{res['mae']:>10,.0f} {res['mape']:>7.1f}%")

best_model = min(results, key=lambda k: results[k]['rmse'])
print(f'\n  ✅ Best model: {best_model} (lowest RMSE)')
print('=' * 65)

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — Test Set Performance (12 weeks)', fontsize=13, fontweight='bold')

model_names = list(results.keys())
bar_colors  = [C[0], C[1], C[2]]
metric_info = [
    ('rmse', 'RMSE (£) — lower=better'),
    ('mae',  'MAE (£) — lower=better'),
    ('mape', 'MAPE (%) — lower=better'),
]

for ax, (metric, title) in zip(axes, metric_info):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(model_names, vals, color=bar_colors, edgecolor='white', width=0.5)
    best_idx = int(np.argmin(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    for bar, val in zip(bars, vals):
        label = f'£{val:,.0f}' if metric != 'mape' else f'{val:.1f}%'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
                label, ha='center', fontsize=9)
    ax.set_title(title)
    ax.set_xticklabels(model_names, rotation=10)

plt.tight_layout()
plt.show()

## Cell 11 — Forecast vs Actual (All 3 Models)
> Each model shown with **95% confidence interval** shaded.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 16))
fig.suptitle('Forecast vs Actual — Test Period (12 weeks) with 95% CI',
             fontsize=14, fontweight='bold')

model_list = [
    ('ARIMA',        C[0], 'ARIMA — Baseline (no seasonality, flat forecast expected)'),
    ('Holt-Winters', C[1], 'Holt-Winters ETS — Captures annual Christmas spike'),
    ('Prophet',      C[2], 'Prophet — Fourier seasonality + UK holidays + CI'),
]

for ax, (name, color, title) in zip(axes, model_list):
    res  = results[name]
    rmse = res['rmse']

    # Training tail
    train_tail = train.tail(20)
    ax.plot(train_tail['ds'], train_tail['y']/1000,
            color='gray', lw=2, alpha=0.7, label='Historical (train)')

    # Actual test
    ax.plot(test['ds'], test['y']/1000,
            color='black', lw=2.5, label='Actual (test)', zorder=5)

    # Predicted
    ax.plot(test['ds'], res['predictions']/1000,
            color=color, lw=2, linestyle='--',
            label=f'{name} forecast  (RMSE = £{rmse:,.0f})')

    # CI shading
    ax.fill_between(test['ds'],
                    res['lower']/1000, res['upper']/1000,
                    alpha=0.2, color=color, label='95% Confidence Interval')

    ax.axvline(test['ds'].iloc[0], color='gray', lw=1, linestyle=':')
    winner_tag = '  ✅ BEST' if name == best_model else ''
    ax.set_title(f'{title}{winner_tag}  |  RMSE: £{rmse:,.0f}')
    ax.set_ylabel('Revenue (£K)')
    ax.legend(fontsize=9, loc='upper left')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Cell 12 — Generate 8-Week Future Forecast (Best Model)
> Retrain on **full dataset** (train + test combined), then forecast the next 8 weeks.

In [ ]:
FORECAST_WEEKS = 8

print(f'Generating {FORECAST_WEEKS}-week future forecast using: {best_model}')
print(f'Retraining on FULL dataset ({len(ts)} weeks)\n')

if best_model == 'Prophet':
    full_prophet = Prophet(
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        seasonality_mode        = 'multiplicative',
        interval_width          = 0.95,
        changepoint_prior_scale = 0.15,
        seasonality_prior_scale = 20,
        n_changepoints          = 20,
    )
    full_prophet.add_country_holidays(country_name='GB')
    full_prophet.fit(ts[['ds', 'y']])
    future_df  = full_prophet.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
    fc_all     = full_prophet.predict(future_df)
    future_fc  = fc_all.tail(FORECAST_WEEKS)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    future_fc.columns = ['Date', 'Forecast', 'Lower_CI', 'Upper_CI']
    full_model = full_prophet
    fc_full_final = fc_all

elif best_model == 'Holt-Winters':
    hw_full = ExponentialSmoothing(
        ts['y'], trend='add', seasonal='mul', seasonal_periods=52,
        damped_trend=True, initialization_method='estimated',
    ).fit(optimized=True, remove_bias=True)
    future_dates = pd.date_range(ts['ds'].max(), periods=FORECAST_WEEKS+1, freq='W')[1:]
    hw_fc        = np.clip(hw_full.forecast(FORECAST_WEEKS).values, 0, None)
    np.random.seed(42)
    sims = hw_full.simulate(FORECAST_WEEKS, repetitions=500, error='mul')
    future_fc = pd.DataFrame({
        'Date'    : future_dates,
        'Forecast': hw_fc,
        'Lower_CI': np.clip(np.percentile(sims, 2.5,  axis=1), 0, None),
        'Upper_CI': np.clip(np.percentile(sims, 97.5, axis=1), 0, None),
    })
    full_model = hw_full
    fc_full_final = None

else:  # ARIMA
    full_arima = auto_arima(ts['y'], seasonal=False, stepwise=True, suppress_warnings=True)
    preds, ci  = full_arima.predict(n_periods=FORECAST_WEEKS, return_conf_int=True, alpha=0.05)
    future_dates = pd.date_range(ts['ds'].max(), periods=FORECAST_WEEKS+1, freq='W')[1:]
    future_fc = pd.DataFrame({
        'Date'    : future_dates,
        'Forecast': np.clip(preds, 0, None),
        'Lower_CI': np.clip(ci[:, 0], 0, None),
        'Upper_CI': np.clip(ci[:, 1], 0, None),
    })
    full_model = full_arima
    fc_full_final = None

print('8-Week Revenue Forecast:')
print('-' * 60)
for _, row in future_fc.iterrows():
    print(f"  {row['Date'].strftime('%Y-%m-%d')}  "
          f"£{row['Forecast']:>10,.0f}  "
          f"(95% CI: £{row['Lower_CI']:,.0f} – £{row['Upper_CI']:,.0f})")
print(f"\n  Total 8-week forecast: £{future_fc['Forecast'].sum():,.0f}")

## Cell 13 — Visualize Final 8-Week Forecast + Confidence Interval

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Historical — last 26 weeks
hist = ts.tail(26)
ax.fill_between(hist['ds'], hist['y']/1000, alpha=0.08, color=C[0])
ax.plot(hist['ds'], hist['y']/1000, color=C[0], lw=2.5, label='Historical (last 26 wks)')

# Connect historical to forecast
last_actual = ts.iloc[-1]
ax.plot([last_actual['ds'], future_fc['Date'].iloc[0]],
        [last_actual['y']/1000, future_fc['Forecast'].iloc[0]/1000],
        color=C[2], lw=1.5, linestyle='--', alpha=0.4)

# Forecast line
ax.plot(future_fc['Date'], future_fc['Forecast']/1000,
        color=C[2], lw=2.5, linestyle='--', marker='o',
        markersize=8, zorder=5, label=f'{best_model} Forecast (next {FORECAST_WEEKS} wks)')

# CI band
ax.fill_between(future_fc['Date'],
                future_fc['Lower_CI']/1000, future_fc['Upper_CI']/1000,
                alpha=0.22, color=C[2], label='95% Confidence Interval')

# Forecast start marker
ax.axvline(future_fc['Date'].iloc[0], color='gray', lw=1.5, linestyle=':', alpha=0.7)

# Value labels
for _, row in future_fc.iterrows():
    ax.annotate(f"£{row['Forecast']/1000:.0f}K",
                (row['Date'], row['Forecast']/1000),
                textcoords='offset points', xytext=(0, 11),
                ha='center', fontsize=8, color=C[2], fontweight='bold')

ax.set_title(
    f'8-Week Revenue Forecast — {best_model} Model\n'
    f'Total predicted: £{future_fc["Forecast"].sum():,.0f}  |  '
    f'Range: £{future_fc["Lower_CI"].sum():,.0f} – £{future_fc["Upper_CI"].sum():,.0f}',
    fontsize=12, fontweight='bold'
)
ax.set_ylabel('Revenue (£K)')
ax.legend(loc='upper left')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Cell 14 — Seasonality Analysis
> Breaks the forecast into **Trend + Seasonal + Holiday** components.  
> Shows exactly **when in the year** revenue peaks — confirming the model learned the Christmas spike.

In [ ]:
if best_model == 'Prophet' and fc_full_final is not None:
    print('Prophet decomposes the forecast into components:')
    print('  Trend    — long-term growth direction')
    print('  Yearly   — the annual seasonal pattern (Christmas peak)')
    print('  Holidays — specific UK bank holiday effects')
    print()

    # Component chart (built-in)
    fig = full_model.plot_components(fc_full_final)
    plt.suptitle('Prophet Forecast Components', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # Monthly seasonality bar chart
    yc = fc_full_final[['ds', 'yearly']].copy()
    yc['month'] = yc['ds'].dt.month
    monthly_seasonal = yc.groupby('month')['yearly'].mean() * 100

    fig2, ax2 = plt.subplots(figsize=(12, 4))
    bar_colors = [C[2] if m in [11, 12] else C[0] for m in range(1, 13)]
    bars = ax2.bar(range(1, 13), monthly_seasonal, color=bar_colors, alpha=0.8, edgecolor='white')
    ax2.axhline(0, color='gray', lw=1, linestyle='--')
    ax2.set_xticks(range(1, 13))
    ax2.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'])
    ax2.set_title('Monthly Seasonality Effect (% of trend level)')
    ax2.set_ylabel('Seasonal Effect (%)')
    for bar, val in zip(bars, monthly_seasonal):
        ypos = bar.get_height() + 0.3 if val >= 0 else bar.get_height() - 1.5
        ax2.text(bar.get_x() + bar.get_width()/2, ypos,
                 f'{val:.1f}%', ha='center', fontsize=8)
    plt.tight_layout()
    plt.show()

    peak_month = monthly_seasonal.idxmax()
    peak_val   = monthly_seasonal.max()
    import calendar
    print(f'Peak seasonal month : {calendar.month_name[peak_month]} (+{peak_val:.1f}% above trend)')
    print(f'Christmas premium   : revenue in Nov-Dec is ~{peak_val:.0f}% higher than the trend baseline')

elif best_model == 'Holt-Winters':
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))
    fitted = full_model.fittedvalues
    axes[0].plot(ts['ds'], ts['y']/1000, color=C[0], lw=1.5, alpha=0.6, label='Actual')
    axes[0].plot(ts['ds'], fitted.values/1000, color=C[1], lw=2, label='Holt-Winters fitted')
    axes[0].set_title('Holt-Winters — In-Sample Fit vs Actual')
    axes[0].set_ylabel('Revenue (£K)')
    axes[0].legend()
    resid = ts['y'].values - fitted.values
    axes[1].bar(range(len(ts)), resid/1000, color=C[3], alpha=0.6, width=1)
    axes[1].axhline(0, color='gray', lw=1, linestyle='--')
    axes[1].set_title('Residuals (Actual − Fitted)')
    axes[1].set_ylabel('Residual (£K)')
    axes[1].set_xlabel('Week #')
    plt.tight_layout()
    plt.show()
else:
    print('ARIMA is the best model — no seasonal decomposition (ARIMA has no seasonal component).')
    print('This is unusual. Consider checking data quality or re-running Prophet.')

## Cell 15 — Save All Outputs

In [ ]:
import joblib, os, pickle

SAVE_MODEL = SAVE + 'models/'
os.makedirs(SAVE_MODEL, exist_ok=True)

# 1. Forecast CSV (→ Power BI + Inventory Agent)
future_fc.to_csv(SAVE + 'data_forecast.csv', index=False)
print(f'✓ data_forecast.csv          — {len(future_fc)} weeks of forecast')

# 2. Test predictions with CI (all 3 models)
test_results = test[['ds', 'y']].copy()
test_results.columns = ['Date', 'Actual']
for name in results:
    test_results[name]             = results[name]['predictions']
    test_results[f'{name}_Lower']  = results[name]['lower']
    test_results[f'{name}_Upper']  = results[name]['upper']
test_results.to_csv(SAVE + 'data_forecast_test.csv', index=False)
print(f'✓ data_forecast_test.csv     — all model predictions + 95% CI')

# 3. Model scores
scores_df = pd.DataFrame([
    {'Model': k, 'RMSE': v['rmse'], 'MAE': v['mae'], 'MAPE': v['mape'], 'Best': k == best_model}
    for k, v in results.items()
])
scores_df.to_csv(SAVE + 'data_forecast_scores.csv', index=False)
print(f'✓ data_forecast_scores.csv   — RMSE/MAE/MAPE all models')

# 4. Best model artifact
if best_model == 'Prophet':
    with open(SAVE_MODEL + 'forecast_prophet.pkl', 'wb') as f:
        pickle.dump(full_model, f)
    print(f'✓ models/forecast_prophet.pkl')
elif best_model == 'Holt-Winters':
    joblib.dump(full_model, SAVE_MODEL + 'forecast_holtwinters.pkl')
    print(f'✓ models/forecast_holtwinters.pkl')
else:
    joblib.dump(full_model, SAVE_MODEL + 'forecast_arima.pkl')
    print(f'✓ models/forecast_arima.pkl')

print(f'\nSaved to: {SAVE}')
print('\nDownstream consumers:')
print('  Power BI  → data_forecast.csv  (Forecast page — actual vs predicted line chart)')
print('  Agent 2   → LangGraph Demand Forecasting output')
print('  Agent 6   → Inventory Reorder uses forecast qty to decide reorder volume')

## Cell 16 — Summary

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════╗
║       MODEL 2 — DEMAND FORECASTING  (v2, FIXED)             ║
╠══════════════════════════════════════════════════════════════╣
║  Models:                                                     ║
║    ARIMA        — baseline, no seasonality (flat)            ║
║    Holt-Winters — multiplicative ETS, m=52 (seasonal ✓)     ║
║    Prophet      — Fourier series + UK holidays + CI (best)   ║
║                                                              ║
║  Input  : data_ts_weekly.csv  (106 weeks)                    ║
║  Output : data_forecast.csv   (8-week forecast + 95% CI)     ║
╠══════════════════════════════════════════════════════════════╣''')

print(f"║  {'Model':<18} {'RMSE':>10} {'MAPE':>7}  {'Winner':>8}  ║")
print('║' + '─'*62 + '║')
for name, res in results.items():
    winner = '✅ BEST' if name == best_model else ''
    print(f"║  {name:<18} £{res['rmse']:>9,.0f} {res['mape']:>6.1f}%  {winner:>8}  ║")

print('''
╠══════════════════════════════════════════════════════════════╣
║  Key fixes vs v1:                                            ║
║    ✓ SARIMA(m=52) → Holt-Winters ETS                        ║
║      SARIMA needs 3+ cycles; 106 wks = only ~1.8 cycles     ║
║    ✓ 95% CI on ALL 3 models (not just Prophet)               ║
║    ✓ STL decomposition — visualises trend + seasonality      ║
║    ✓ Prophet tuned: changepoint + seasonality priors         ║
║    ✓ Monthly seasonality bar chart                           ║
║    ✓ Forecast value labels on final chart                    ║
║                                                              ║
║  NEXT → Model 3: Churn Prediction                            ║
║  Files: data_rfm.csv | Models: Random Forest + XGBoost       ║
╚══════════════════════════════════════════════════════════════╝
''')